<a href="https://colab.research.google.com/github/Sucheta-afk/Graphs-for-Emotion-Recognition/blob/main/Build_graph_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torch-scatter torch-sparse -q -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 682.4/682.4 kB 11.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 44.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.4 MB/s eta 0:00:00


In [3]:
import os, time, pickle
import numpy as np
import scipy.io as sio
import torch
from torch_geometric.data import Data
from scipy.signal import butter, filtfilt, hilbert

In [4]:
BANDS = {
    'delta': (1, 4),
    'theta': (4, 8),
    'alpha': (8, 14),
    'beta':  (14, 31),
    'gamma': (31, 50)
}
from scipy.signal import butter, filtfilt


def bandpass_filter(eeg, lowcut, highcut, fs=200, order=4):
    """
    eeg   : shape (n_channels, n_samples)
    returns filtered eeg of same shape
    """
    nyq = fs / 2
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    # apply filter to each channel independently
    filtered = filtfilt(b, a, eeg, axis=1)
    return filtered
from scipy.signal import hilbert

# HAD TO MODIFY FUNCTION TO REDUCE GRAPH GENERATION TIME (5HRS TO 1HR)
def compute_plv_matrix(filtered_eeg):
    """
    filtered_eeg : shape (n_channels, n_samples) — already band-pass filtered
    returns      : plv_matrix of shape (n_channels, n_channels)
    """
    n_channels = filtered_eeg.shape[0]

    # Step 1: extract instantaneous phase for every channel
    analytic = hilbert(filtered_eeg, axis=1)       # (62, N_samples) complex
    phase = np.angle(analytic)                      # (62, N_samples) real, in radians

    # vectorized — no for loop over pairs
    exp_phase = np.exp(1j * phase)  # (62, N_samples)
    plv_matrix = np.abs(exp_phase @ exp_phase.conj().T) / filtered_eeg.shape[1]

    np.fill_diagonal(plv_matrix, 1.0)  # a channel is perfectly locked with itself
    return plv_matrix


def compute_averaged_plv(raw_eeg, fs=200):
    """
    raw_eeg : shape (62, N_samples)
    returns : averaged PLV matrix of shape (62, 62)
    """
    plv_all_bands = []

    for band_name, (low, high) in BANDS.items():
        filtered = bandpass_filter(raw_eeg, low, high, fs=fs)
        plv = compute_plv_matrix(filtered)
        plv_all_bands.append(plv)

    # stack → (5, 62, 62), then average across bands → (62, 62)
    averaged_plv = np.mean(np.stack(plv_all_bands, axis=0), axis=0)
    return averaged_plv
def plv_to_edge_index(plv_matrix, threshold=0.5):
    """
    plv_matrix : (62, 62) numpy array
    threshold  : keep edges where PLV > threshold
    returns    : edge_index tensor of shape (2, E)
                 edge_weight tensor of shape (E,)
    """
    n = plv_matrix.shape[0]
    source_nodes = []
    target_nodes = []
    edge_weights = []

    for i in range(n):
        for j in range(i+1, n):   # upper triangle only
            if plv_matrix[i, j] > threshold:
                # add both directions (undirected graph)
                source_nodes.extend([i, j])
                target_nodes.extend([j, i])
                edge_weights.extend([plv_matrix[i, j],
                                     plv_matrix[i, j]])

    edge_index = torch.tensor([source_nodes, target_nodes],
                               dtype=torch.long)
    edge_weight = torch.tensor(edge_weights, dtype=torch.float)

    return edge_index, edge_weight
from torch_geometric.data import Data

def build_graph(raw_eeg, de_features, label, subject_id, trial_id, threshold=0.4):
    """
    raw_eeg     : (62, N_samples) — for PLV computation
    de_features : (62, 5) — node features from eeg_feature_smooth
    label       : int — emotion label (0/1/2/3)
    subject_id  : int
    trial_id    : int
    returns     : PyG Data object
    """
    # compute PLV graph
    plv_matrix = compute_averaged_plv(raw_eeg)
    edge_index, edge_weight = plv_to_edge_index(plv_matrix, threshold)

    # node features
    x = torch.tensor(de_features, dtype=torch.float)   # (62, 5)

    # label
    y = torch.tensor([label], dtype=torch.long)

    data = Data(
        x=x,
        edge_index=edge_index,
        edge_attr=edge_weight,
        y=y,
        subject_id=subject_id,
        trial_id=trial_id
    )
    return data

In [5]:

TRIAL_LABELS = {
    '1': [1,2,3,0,2,0,0,1,0,1,2,1,1,1,2,3,2,2,3,3,0,3,0,3],
    '2': [2,1,3,0,0,2,0,2,3,3,2,3,2,0,1,1,2,1,0,3,0,1,3,1],
    '3': [1,2,2,1,3,3,3,1,1,2,1,0,2,3,3,0,2,3,0,0,2,0,1,0]
}

print('Number of trial labels:', len(TRIAL_LABELS))

Number of trial labels: 3


In [6]:
from google.colab import drive
drive.mount('/content/drive')

raw_base = '/content/drive/MyDrive/SEED-IV/eeg_raw_data'
feat_base = '/content/drive/MyDrive/SEED-IV/eeg_feature_smooth'

for session in ['1', '2', '3']:
    raw_files = sorted(os.listdir(os.path.join(raw_base, session)))
    feat_files = sorted(os.listdir(os.path.join(feat_base, session)))
    print(f'Session {session}: {len(raw_files)} raw files, {len(feat_files)} feature files')

Mounted at /content/drive
Session 1: 15 raw files, 15 feature files
Session 2: 15 raw files, 15 feature files
Session 3: 15 raw files, 15 feature files


In [18]:
def load_subject_session(subject_idx, session, raw_base, feat_base, threshold=0.4):
    """
    subject_idx : 0-based index into the sorted file list
    session     : '1', '2', or '3'
    returns     : list of PyG Data objects, one per trial
    """
    raw_path  = os.path.join(raw_base, session)
    feat_path = os.path.join(feat_base, session)

    raw_files  = sorted(os.listdir(raw_path))
    feat_files = sorted(os.listdir(feat_path))

    raw_mat  = sio.loadmat(os.path.join(raw_path,  raw_files[subject_idx]))
    feat_mat = sio.loadmat(os.path.join(feat_path, feat_files[subject_idx]))

    graphs = []
    for trial_idx in range(24):
        trial_num = trial_idx + 1  # keys are 1-indexed

        # raw EEG for this trial
        # replace hardcoded 'tyc_eeg1' with:

        # CHANGED THE CODE HERE BCS, WE FOUND OUT THAT EVERY SUBJECT HAS A DIFFERENT NAME WITH A COMMON EEG IN IT.
        raw_keys = [k for k in raw_mat.keys() if k.endswith(f'eeg{trial_num}') and not k.startswith('__')]
        raw_key = raw_keys[0]

        # DE features for this trial
        feat_keys = [k for k in feat_mat.keys() if k.endswith(f'LDS{trial_num}') and not k.startswith('__')]
        feat_key = feat_keys[0]

        raw_eeg    = raw_mat[raw_key]          # (62, N_samples)
        de_trial   = feat_mat[feat_key]        # (62, T, 5)
        de_mean    = de_trial.mean(axis=1)     # (62, 5) — average over time windows

        label = TRIAL_LABELS[session][trial_idx]

        subject_id = subject_idx + 1

        graph = build_graph(raw_eeg, de_mean, label,
                            subject_id=subject_id,
                            trial_id=trial_num,
                            threshold=threshold)
        graphs.append(graph)

    return graphs

testing on subject 0 session1

In [19]:
start = time.time()
test_graphs = load_subject_session(0, '1', raw_base, feat_base)
elapsed = time.time() - start

print(f'Loaded {len(test_graphs)} graphs in {elapsed:.1f}s')
print('Sample graph:', test_graphs[0])
print('x shape:', test_graphs[0].x.shape)       # (62, 5)
print('y:', test_graphs[0].y)
print('subject_id:', test_graphs[0].subject_id)

Loaded 24 graphs in 62.8s
Sample graph: Data(x=[62, 5], edge_index=[2, 1058], edge_attr=[1058], y=[1], subject_id=1, trial_id=1)
x shape: torch.Size([62, 5])
y: tensor([1])
subject_id: 1


In [20]:
time_per_session = elapsed   # from Step 4 test
total_estimated = time_per_session * 15 * 3 / 3600
print(f'Estimated total time: {total_estimated:.1f} hours')

Estimated total time: 0.8 hours


In [21]:
save_dir = '/content/drive/MyDrive/SEED-IV/pyg_graphs'
os.makedirs(save_dir, exist_ok=True)
print('Save directory ready:', save_dir)

Save directory ready: /content/drive/MyDrive/SEED-IV/pyg_graphs


In [22]:
def run_full_dataset(raw_base, feat_base, save_dir, threshold=0.4):
    total_subjects = 15
    sessions = ['1', '2', '3']

    for session in sessions:
        for subject_idx in range(total_subjects):
            subject_id = subject_idx + 1
            filename = f'subject_{subject_id:02d}_session_{session}.pkl'
            save_path = os.path.join(save_dir, filename)

            # RESUME SUPPORT — skip if already saved
            if os.path.exists(save_path):
                print(f'[SKIP] {filename} already exists')
                continue

            print(f'[PROCESSING] Subject {subject_id}, Session {session}...')
            start = time.time()

            try:
                graphs = load_subject_session(subject_idx, session,
                                              raw_base, feat_base, threshold)
                with open(save_path, 'wb') as f:
                    pickle.dump(graphs, f)

                elapsed = time.time() - start
                print(f'[DONE] {filename} — {len(graphs)} graphs — {elapsed:.1f}s')

            except Exception as e:
                print(f'[ERROR] Subject {subject_id}, Session {session}: {e}')
                continue  # skip and move on, don't crash the whole loop

In [23]:
run_full_dataset(raw_base, feat_base, save_dir, threshold=0.4)

[SKIP] subject_01_session_1.pkl already exists
[PROCESSING] Subject 2, Session 1...
[DONE] subject_02_session_1.pkl — 24 graphs — 61.7s
[PROCESSING] Subject 3, Session 1...
[DONE] subject_03_session_1.pkl — 24 graphs — 65.3s
[PROCESSING] Subject 4, Session 1...
[DONE] subject_04_session_1.pkl — 24 graphs — 62.6s
[PROCESSING] Subject 5, Session 1...
[DONE] subject_05_session_1.pkl — 24 graphs — 62.2s
[PROCESSING] Subject 6, Session 1...
[DONE] subject_06_session_1.pkl — 24 graphs — 83.1s
[PROCESSING] Subject 7, Session 1...
[DONE] subject_07_session_1.pkl — 24 graphs — 78.7s
[PROCESSING] Subject 8, Session 1...
[DONE] subject_08_session_1.pkl — 24 graphs — 63.3s
[PROCESSING] Subject 9, Session 1...
[DONE] subject_09_session_1.pkl — 24 graphs — 62.1s
[PROCESSING] Subject 10, Session 1...
[DONE] subject_10_session_1.pkl — 24 graphs — 60.8s
[PROCESSING] Subject 11, Session 1...
[DONE] subject_11_session_1.pkl — 24 graphs — 62.7s
[PROCESSING] Subject 12, Session 1...
[DONE] subject_12_sessi

In [24]:
saved = os.listdir(save_dir)
print(f'Saved so far: {len(saved)} / 45 files')

Saved so far: 45 / 45 files


In [25]:
with open(os.path.join(save_dir, 'subject_01_session_1.pkl'), 'rb') as f:
    loaded = pickle.load(f)

print(f'Graphs in file: {len(loaded)}')       # should be 24
print('First graph:', loaded[0])
print('Labels:', [g.y.item() for g in loaded])  # 24 labels

Graphs in file: 24
First graph: Data(x=[62, 5], edge_index=[2, 1058], edge_attr=[1058], y=[1], subject_id=1, trial_id=1)
Labels: [1, 2, 3, 0, 2, 0, 0, 1, 0, 1, 2, 1, 1, 1, 2, 3, 2, 2, 3, 3, 0, 3, 0, 3]


In [17]:
raw_files = sorted(os.listdir(os.path.join(raw_base, '1')))
mat2 = sio.loadmat(os.path.join(raw_base, '1', raw_files[1]))
[k for k in mat2.keys() if not k.startswith('__')]

['whh_eeg1',
 'whh_eeg2',
 'whh_eeg3',
 'whh_eeg4',
 'whh_eeg5',
 'whh_eeg6',
 'whh_eeg7',
 'whh_eeg8',
 'whh_eeg9',
 'whh_eeg10',
 'whh_eeg11',
 'whh_eeg12',
 'whh_eeg13',
 'whh_eeg14',
 'whh_eeg15',
 'whh_eeg16',
 'whh_eeg17',
 'whh_eeg18',
 'whh_eeg19',
 'whh_eeg20',
 'whh_eeg21',
 'whh_eeg22',
 'whh_eeg23',
 'whh_eeg24']

subject 1's file names start with 'tyc_eeg'  
subject 2's 'whh_eeg'

In [26]:
index = []
for fname in sorted(os.listdir(save_dir)):
    if not fname.endswith('.pkl'):
        continue
    fpath = os.path.join(save_dir, fname)
    with open(fpath, 'rb') as f:
        graphs = pickle.load(f)
    for g in graphs:
        index.append({
            'file': fname,
            'subject_id': g.subject_id,
            'trial_id': g.trial_id,
            'label': g.y.item(),
            'num_edges': g.num_edges
        })

import pandas as pd
df = pd.DataFrame(index)
df.to_csv(os.path.join(save_dir, 'dataset_index.csv'), index=False)
print(df.head(10))
print('Total graphs:', len(df))   # should be 1080

                       file  subject_id  trial_id  label  num_edges
0  subject_01_session_1.pkl           1         1      1       1058
1  subject_01_session_1.pkl           1         2      2       1060
2  subject_01_session_1.pkl           1         3      3        980
3  subject_01_session_1.pkl           1         4      0       1036
4  subject_01_session_1.pkl           1         5      2       1060
5  subject_01_session_1.pkl           1         6      0       1126
6  subject_01_session_1.pkl           1         7      0       1112
7  subject_01_session_1.pkl           1         8      1       1056
8  subject_01_session_1.pkl           1         9      0       1038
9  subject_01_session_1.pkl           1        10      1       1058
Total graphs: 1080


#### Day 4 Report  
45 pkl files - each containing 24 graphs  
**Total - 1080 graphs**  
Generated a **dataset_index.csv** that maps every graph to its subject ID, session, and emotion label for easy loading during the training phase.  


